# Experiment 002: BM25 Baseline (BM25S)

**Date:** January 26, 2026  
**Objective:** Establish BM25S baseline with Identity query enhancement (no enhancement)  
**Expected Results:**
- Recall@100: ~0.860 (96.8% of Pyserini target 0.889)
- NDCG@10: ~0.461 (95.8% of Pyserini target 0.481)
- Recall@10: (thesis baseline)

**Query Enhancement:** Identity (returns query unchanged)

**Note:** This experiment uses pre-built BM25S index stored in Google Drive

## Setup

### Step 1: Clone Repository and Install Dependencies

In [ ]:
# Clone repository
!git clone https://github.com/Osmanoor/graduation.git
%cd graduation/arabic-rag-query-enhancement

# Install Python dependencies (no Java needed!)
!pip install -q bm25s PyStemmer nltk pytrec_eval

print("\n" + "="*60)
print("✓ Installation complete")
print("="*60)
print("⚠️ IMPORTANT: Restart runtime now!")
print("   1. Click 'Runtime' → 'Restart runtime'")
print("   2. Then run cells starting from 'Step 2' below")
print("="*60)

### Step 2: Mount Google Drive and Configure Environment (Run After Restart)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to project directory
%cd /content/graduation/arabic-rag-query-enhancement

# Configure environment
import os
import sys

# Add src to path
sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

print("\n✓ Environment configured")
print("✓ Ready to run experiment")

### Step 3: Setup Symbolic Links to Google Drive Index

In [ ]:
# Create data directory
!mkdir -p data/miracl_ar

# Create symbolic links to Google Drive
# IMPORTANT: Update these paths to match your Google Drive structure
drive_base = "/content/drive/MyDrive/graduation_project"  # Update this path!

# Link to BM25S index
!ln -sf "{drive_base}/bm25s_index" data/miracl_ar/bm25s_index

# Link to corpus IDs
!ln -sf "{drive_base}/corpus_ids.pkl" data/miracl_ar/corpus_ids.pkl

# Verify links
print("Verifying index files...")
print(f"Index exists: {os.path.exists('data/miracl_ar/bm25s_index')}")
print(f"Corpus IDs exist: {os.path.exists('data/miracl_ar/corpus_ids.pkl')}")

if not os.path.exists('data/miracl_ar/bm25s_index'):
    print("\n⚠️ ERROR: Index not found!")
    print("Please update the 'drive_base' path above to match your Google Drive structure")
else:
    print("\n✓ Index files linked successfully")

## Import Modules

In [ ]:
from src.utils.data_loader import MIRACLDataLoader
from src.retrievers.bm25 import BM25SRetriever
from src.enhancers.base import IdentityEnhancer
from src.evaluation.metrics import RetrievalEvaluator, save_results, save_metrics, print_metrics

from tqdm.notebook import tqdm

print("✓ Modules imported")

## Load Data

In [ ]:
# Load MIRACL Arabic dev set
data_loader = MIRACLDataLoader(language="ar", split="dev")
topics, qrels = data_loader.load_all()

print(f"\nDataset Statistics:")
print(f"  Queries: {len(topics)}")
print(f"  Qrels: {len(qrels)}")

# Show sample
sample_qid = list(topics.keys())[0]
print(f"\nSample Query:")
print(f"  ID: {sample_qid}")
print(f"  Text: {topics[sample_qid]['title']}")
print(f"  Relevant docs: {len(qrels.get(sample_qid, {}))}")

## Initialize Components

In [ ]:
# Initialize query enhancer (Identity = no enhancement)
enhancer = IdentityEnhancer()
print("✓ Query enhancer: Identity (baseline)")

# Initialize retriever (loads from Google Drive)
retriever = BM25SRetriever(
    index_path="data/miracl_ar/bm25s_index",
    corpus_ids_path="data/miracl_ar/corpus_ids.pkl"
)

# Initialize evaluator
evaluator = RetrievalEvaluator(qrels)
print("✓ Evaluator initialized")

## Run Experiment

In [ ]:
print("="*60)
print("EXPERIMENT 002: BM25 Baseline (Identity Enhancement)")
print("="*60)

# Prepare queries
query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

# Apply query enhancement (Identity = no change)
print(f"\nApplying query enhancement...")
enhanced_queries = enhancer.enhance_batch(query_texts, query_ids)
print(f"✓ Enhanced {len(enhanced_queries)} queries")

# Verify enhancement (should be identical for Identity)
print(f"\nVerification:")
print(f"  Original: {query_texts[0]}")
print(f"  Enhanced: {enhanced_queries[0]}")
print(f"  Same: {query_texts[0] == enhanced_queries[0]}")

In [ ]:
# Run retrieval
print(f"\nRunning retrieval for {len(enhanced_queries)} queries...")
print("This will take ~3-5 minutes")

search_results = retriever.search(enhanced_queries, k=100, show_progress=True)

print(f"✓ Retrieval complete")

In [ ]:
# Format results for evaluation
print("\nFormatting results...")
results = {}

for i, qid in enumerate(tqdm(query_ids, desc="Processing")):
    results[qid] = {}
    for docid, score in search_results[i]:
        results[qid][docid] = score

print(f"✓ Formatted {len(results)} query results")

## Evaluate Results

In [ ]:
# Compute metrics
print("\nEvaluating...")
metrics = evaluator.evaluate(results)

# Print results
print_metrics(metrics, "EXPERIMENT 002: BM25 Baseline Results")

# Compare with expected
print("\nComparison with Pyserini Target:")
print(f"  Recall@100: {metrics['recall_100']:.4f} (Target: ~0.889)")
print(f"  NDCG@10:    {metrics['ndcg_cut_10']:.4f} (Target: ~0.481)")

# Calculate achievement
recall100_achievement = (metrics['recall_100'] / 0.889) * 100
ndcg10_achievement = (metrics['ndcg_cut_10'] / 0.481) * 100

print(f"\nAchievement:")
print(f"  Recall@100: {recall100_achievement:.2f}%")
print(f"  NDCG@10:    {ndcg10_achievement:.2f}%")

## Save Results

In [ ]:
import os

# Create output directory
output_dir = "results/baseline_bm25"
os.makedirs(output_dir, exist_ok=True)

# Save results in TREC format
save_results(
    results,
    f"{output_dir}/exp_002_baseline_bm25.txt",
    run_name="exp_002_identity"
)

# Save metrics
save_metrics(
    metrics,
    f"{output_dir}/exp_002_metrics.json"
)

print("\n✓ All results saved")

## Summary

**Experiment:** 002 - BM25 Baseline (Identity Enhancement)  
**Status:** Complete  

**Results:**
- Recall@10: [Will be filled after run]
- Recall@100: [Will be filled after run]
- NDCG@10: [Will be filled after run]
- MRR: [Will be filled after run]

**Next Steps:**
1. Document results in `experiments/exp_002_baseline_bm25.md`
2. Compare with Dense baseline (Exp 001)
3. Analyze complementary strengths
4. Prepare for Query Enhancement experiments (Exp 003+)